[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/125/blob/main/06_series/06_1_Labels_and_Lookups.ipynb)

# 06.1: Labels and Lookups

So far you have stored related data in lists. If you had a list of college majors and a separate list of the typical salary for each major, you would rely on position to connect them: the salary at position 0 belongs to the major at position 0. This notebook shows where that approach breaks down, and it introduces the pandas **Series**, which keeps each value together with a label so that the two stay together when you sort. By the end you will be able to create a series and look up values in it by label and by position.

This notebook opens in Google Colab. If Colab warns that the notebook was not authored by Google, click **Run anyway**. Your changes are not kept when you close the tab unless you choose **File > Save a copy in Drive**, so do that if you want to keep your work.

## The data

We will use data on recent college graduates in 173 majors, collected from U.S. Census data around 2010-2012. The next cell uses a library called **pandas** to read the file and pull out two lists. pandas is not part of Python's standard library, but it is already installed in Colab, so all you need to do is import it. `import pandas as pd` loads the library and gives it the short name `pd`, which is what nearly everyone calls it. You will learn how `read_csv` works in module 07. For now, all that matters is that you end up with two ordinary Python lists.

In [ ]:
import pandas as pd

majors = pd.read_csv("https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_recent_grads.csv")
major_list = majors["Major"].to_list()
median_list = majors["Median"].to_list()

In [ ]:
print(major_list[:3])
print(median_list[:3])
print(len(major_list), len(median_list))

`major_list` holds the names and `median_list` holds the median salary in dollars for full-time workers with that major. The **median** is the middle value: half of the workers earned more and half earned less. The first major is PETROLEUM ENGINEERING with a median of 110000, and both lists have 173 items.

## The problem with parallel lists

Lists like these are called parallel lists, because the only thing connecting them is that matching items sit at the same position. Suppose you want the salaries in order. The built-in function `sorted()` returns a new sorted list and leaves the original alone, unlike the `.sort()` method, which changes the list itself.

In [ ]:
sorted_medians = sorted(median_list)
print(major_list[0], sorted_medians[0])

This pairs PETROLEUM ENGINEERING with 22000, which is wrong: the real value is 110000, and 22000 belongs to LIBRARY SCIENCE. Python did not raise an error, because it has no way to know that the two lists were connected, so anything you calculated from these lists after sorting would be wrong without any warning.

## What is a series?

A **Series** is a pandas object that stores a column of values, with a label attached to each value. The labels are called the **index**. To create one, give `pd.Series()` the values, and use the `index=` argument for the labels.

In [ ]:
salaries = pd.Series(median_list, index=major_list)
salaries

pandas shows two columns. The left one is the index (the major names) and the right one is the values (the salaries). The last line gives the length, 173, and the **dtype**, which is the data type of the values. `int64` is the name for whole numbers, and you will also see `float64` for numbers with decimal places. pandas leaves out the middle rows, so that a long series does not fill your screen. Checking the type confirms that this is a new kind of object and not a list.

In [ ]:
type(salaries)

The output says `pandas.core.series.Series`, the full name of the Series type. Now we can repeat the sort. `.sort_values()` returns a new series in order of its values.

In [ ]:
salaries.sort_values()

The top row is now LIBRARY SCIENCE at 22000, and PETROLEUM ENGINEERING with 110000 is at the bottom. Each salary moved together with its label, so the mismatch from the previous section cannot happen here. Sorting did not change `salaries` itself, which is why the next cell still shows PETROLEUM ENGINEERING first. The sorted series was a new result that we did not store, and notebook 06.2 explains more about this.

In [ ]:
salaries.head(1)

## Other ways to create a series

If you do not supply an index, pandas numbers the values starting at 0. These numbers are still labels, and they stay with their values just as the major names did.

In [ ]:
quiz_scores = pd.Series([88, 95, 71])
quiz_scores

The labels here are 0, 1 and 2, and the values are 88, 95 and 71. A dictionary is another natural source for a series, because it already pairs labels with values: the keys become the index and the values become the data.

In [ ]:
fruit_weights = pd.Series({"apple": 180, "banana": 120, "cherry": 15})
fruit_weights

pandas works out the dtype from the values you give it, which is why these came out as `int64`. You can also state the dtype yourself with the `dtype=` argument, and you must do so if you create an empty series to fill later, because there are no values to work it out from.

In [ ]:
empty_series = pd.Series(dtype="int64")
print(len(empty_series))
print(pd.Series([1, 2, 3], dtype="float64"))

The empty series has a length of 0, and giving `dtype="float64"` stored 1, 2 and 3 as 1.0, 2.0 and 3.0. The dtype describes the values and not the labels, so a series with text labels and whole-number values is still `int64`.

## Labels do not have to be unique or in order

Positions in a list are always 0, 1, 2 and so on, but the labels in a series can be anything. They can be out of order, and the same label can appear more than once. The next series has three apples and two bananas.

In [ ]:
fruit_list = ["apple", "apple", "apple", "banana", "banana"]
weight_list = [180, 120, 15, 650, 450]
fruit_series = pd.Series(weight_list, index=fruit_list)
fruit_series

This is legal, and you will see real data like it. For example, a series of Oscar winners labeled by year has several winners for each year. We will return to what happens when you look up a repeated label.

## Looking at part of a series

A series can be too long to read in one piece. `.head()` shows the first five items and `.tail()` shows the last five. Each one takes an optional number if you want more or fewer. These are also useful for checking that your code did what you intended, the way you might use `print()`. `len()` works the same way it does on a list.

In [ ]:
salaries.head()

In [ ]:
salaries.tail(3)

In [ ]:
len(salaries)

The file happens to be ordered from highest to lowest salary, which is why `.head()` shows engineering majors and `.tail(3)` shows CLINICAL PSYCHOLOGY, COUNSELING PSYCHOLOGY and LIBRARY SCIENCE. The length is 173. You can also get at the two parts of a series separately with `.index` and `.values`. There are no parentheses after these two, because they are stored attributes of the series and not methods.

In [ ]:
salaries.index

In [ ]:
salaries.values

`.index` shows the major names inside `Index([...])`, which is pandas' own type for labels. `.values` shows the salaries inside `array([...])`. An array is a structure much like a list that pandas uses to store numbers, and you can index and loop over it the same way.

## Looking up a value by label: `.loc`

To get one value by its label, use `.loc` with square brackets. Think of it as asking what is stored under this label.

In [ ]:
salaries.loc["NURSING"]

The result is a single number, 48000, and not a series. In Colab it may be displayed as `np.int64(48000)`, which is how pandas' underlying number library shows an integer, and it behaves like the ordinary integer 48000. To look up several labels at once, pass a list of labels, which does give back a series.

In [ ]:
salaries.loc[["NURSING", "ECONOMICS", "STUDIO ARTS"]]

NURSING is 48000, ECONOMICS is 47000 and STUDIO ARTS is 29000, in the order we asked for them. If a label is repeated, `.loc` returns every value with that label as a series, so here you get all three apples.

In [ ]:
fruit_series.loc["apple"]

You will also see code that leaves out `.loc` and writes `salaries["NURSING"]`, which gives the same result when the labels are text. We use `.loc` in this course because it says clearly that you are looking up a label.

If a label does not exist, for example because of a misspelling, pandas raises a `KeyError`, which tells you that the label you typed is not in the index. Labels must match exactly, including upper and lower case. The next cell fails on purpose. If you use Run all, Colab stops at this cell, so run the cells after it yourself.

In [ ]:
salaries.loc["Nursing"]

## Looking up a value by position: `.iloc`

Sometimes you care about position and not about the label. `.iloc` works like indexing a list: it starts at 0, and negative numbers count from the end. The `i` stands for integer. Negative numbers would not make sense with `.loc`, because `.loc` is looking for a label.

In [ ]:
salaries.iloc[0]

In [ ]:
salaries.iloc[-1]

Position 0 gives 110000 (PETROLEUM ENGINEERING) and position -1 gives 22000 (LIBRARY SCIENCE). Using `.iloc` depends on the current order of the series. If the series were sorted differently, position 0 would hold something else, while `.loc["NURSING"]` would still return the value for nursing. So prefer `.loc` whenever you know the label.

## Slicing

Both `.loc` and `.iloc` can take a slice, which returns a shorter series. With `.iloc`, a slice works the way it does on a list: the stop position is **not** included.

In [ ]:
salaries.iloc[10:13]

Positions 10, 11 and 12 give three items: COMPUTER ENGINEERING, AEROSPACE ENGINEERING and BIOMEDICAL ENGINEERING, all at 60000. With `.loc`, you slice from one label to another, and the stop label **is** included. This is convenient, because you may not know which label comes right after the one you want to stop at.

In [ ]:
salaries.loc["COMPUTER ENGINEERING":"BIOMEDICAL ENGINEERING"]

This returns the same three items, including BIOMEDICAL ENGINEERING. Leaving off the stop label continues to the end of the series, so `salaries.loc["CLINICAL PSYCHOLOGY":]` returns the last three items.

## Practice problems

These problems are not collected or graded. They are here so that you can find out whether you can write this code yourself before the reading check and the pair activity. Write your answer in the cell under each problem, run it, and then run the check cell below it, which tells you whether your answer is right. The check cells are collapsed because they contain the answers, so try the problem before you open one.

Run the next two cells first. The first one creates fresh data for the problems, and the second sets up the checks.

In [ ]:
import pandas as pd

majors = pd.read_csv("https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_recent_grads.csv")
major_list = majors["Major"].to_list()
employed_list = majors["Employed"].to_list()
want = pd.Series(employed_list, index=major_list)

In [ ]:
#@title Run this cell to set up the checks
import numpy as np
import pandas as pd


def same(got, want):
    """True when got matches want: same labels and values for a series, same value for a number."""
    try:
        if isinstance(want, pd.Series):
            if not isinstance(got, pd.Series) or list(got.index) != list(want.index):
                return False
            if pd.api.types.is_numeric_dtype(want):
                return bool(np.allclose(got.astype(float), want.astype(float), equal_nan=True))
            return list(got) == list(want)
        if isinstance(want, (list, np.ndarray)):
            return list(got) == list(want)
        if isinstance(want, str):
            return got == want
        if isinstance(got, (pd.Series, pd.DataFrame, list, np.ndarray)):
            return False
        return bool(np.isclose(float(got), float(want)))
    except (TypeError, ValueError):
        return False


def check(number, name, want, hint, unchanged=None):
    """Print feedback for one problem. unchanged is (variable name, its starting value)."""
    if name not in globals():
        print(f"Problem {number}: not yet. No variable named {name} exists. Did you run your answer cell?")
    elif not same(globals()[name], want):
        print(f"Problem {number}: not yet. {hint}")
    elif unchanged is not None and not globals()[unchanged[0]].equals(unchanged[1]):
        print(f"Problem {number}: {name} is right, but {unchanged[0]} changed. Work on a copy, then run the setup cell again.")
    else:
        print(f"Problem {number}: correct.")

### Problem 1
Create a series named `employed` from `major_list` and `employed_list`, which holds the number of employed graduates for each major. Use `major_list` for the index.

In [ ]:
# Problem 1

In [ ]:
#@title Check Problem 1
check(1, "employed", want, "Check that the values come from employed_list and that index= uses major_list.")

### Problem 2
Create a series named `last_two` that holds the last two items of `employed`.

In [ ]:
# Problem 2

In [ ]:
#@title Check Problem 2
check(2, "last_two", want.tail(2), "Look at how .tail() takes an argument.")

### Problem 3
Store the number of employed graduates for the label `'NURSING'` in a variable named `nursing_employed`. It should be a single number.

In [ ]:
# Problem 3

In [ ]:
#@title Check Problem 3
check(3, "nursing_employed", want.loc["NURSING"], "Use .loc with the label, spelled exactly as it is in the index.")

### Problem 4
Store the value at position 4 of `employed` in a variable named `fifth_value`.

In [ ]:
# Problem 4

In [ ]:
#@title Check Problem 4
check(4, "fifth_value", want.iloc[4], "This is a lookup by position, and positions start at 0.")

### Problem 5
Create a series named `two_majors` that holds the items for `'MATHEMATICS'` and `'NURSING'`, in that order.

In [ ]:
# Problem 5

In [ ]:
#@title Check Problem 5
check(5, "two_majors", want.loc[["MATHEMATICS", "NURSING"]], "Pass a list of labels to .loc, which takes two sets of square brackets.")

### Problem 6
Create a series named `cs_to_accounting` that starts at the label `'COMPUTER SCIENCE'` and runs through the label `'ACCOUNTING'`, including `'ACCOUNTING'`.

In [ ]:
# Problem 6

In [ ]:
#@title Check Problem 6
check(6, "cs_to_accounting", want.loc["COMPUTER SCIENCE":"ACCOUNTING"], "Use a slice with .loc, which includes the stop label.")

## What's next

You can now create a series and pull out specific values by label or by position. Looking things up one at a time only helps when you already know what you want, though. Notebook 06.2 shows how to work on all of the values at once without a loop, which lets you ask questions like "which majors have a median above $70,000?"